# Option VIX, CBOE method, all A-share equity-option underlyings - 2026-09-27

**Question:** what is the 30-day implied volatility index of every listed equity-option underlying (SSE and SZSE ETF options, CFFEX index options), computed with the CBOE VIX method, what is the most recent value for the **300** family (SSE `510300.SH`, SZSE `159919.SZ`, CFFEX CSI 300 index options), and does the number pass independent checks?

Method (CBOE white paper, as applied in `plans/architecture/CALCULATION_DESIGN.md` §2):
1. Chain per underlying = `opt_basic.opt_code` (e.g. `OP510300.SH`); strike-adjusted ETF contracts (name ending in "A") excluded.
2. Terms = the two nearest expiries with ≥ 7 days to expiry (near, next).
3. Rate = SHIBOR on the day, linearly interpolated by tenor, converted to continuous compounding.
4. Per term: forward `F = K* + e^{RT}(C − P)` at the strike with smallest |C − P|; `K0` = largest strike ≤ F; OTM strip with the two-consecutive-zeros cut-off; `σ² = (2/T) Σ ΔK/K² e^{RT} Q(K) − (1/T)(F/K0 − 1)²`.
5. `VIX = 100 √{[T₁σ₁² (N₂−N₃₀)/(N₂−N₁) + T₂σ₂² (N₃₀−N₁)/(N₂−N₁)] · N₃₆₅/N₃₀}` with times in minutes.

**Deviations forced by the data** (flagged per row in `quality_flag`): no bid/ask in `opt_daily`, so `Q(K)` = close (settle when close is 0/missing); expiry and observation both at the 15:00 close.

Prototype: these functions move into the `calculation` package in plan 03 (phases 2-4), with tests. Data 2026-08-03 → 2026-09-24 (the latest trading day; 09-25 is the Mid-Autumn holiday).

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from quant_cn.core.config import Config
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.visualization.chart_theme import ChartTheme

START, END = "20260801", "20260924"
MIN_DAYS, N30, N365 = 7, 30 * 1440, 365 * 1440
FAMILY_300 = ["OP510300.SH", "OP159919.SZ", "OP000300.SH"]

config = Config.load()
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)

## 1. Load the inputs from the lake

In [ ]:
chain_all = query.sql(
    """
    SELECT d.trade_date, b.opt_code, b.exchange, b.name, b.ts_code, b.call_put,
           b.exercise_price AS strike, b.maturity_date, d.close, d.settle, d.vol, d.oi
    FROM curated.opt_daily d JOIN curated.opt_basic b USING (ts_code)
    WHERE b.exchange IN ('SSE', 'SZSE', 'CFFEX') AND d.trade_date BETWEEN ? AND ?
    """,
    [START, END],
)
chain_all["days"] = (pd.to_datetime(chain_all["maturity_date"]) - pd.to_datetime(chain_all["trade_date"])).dt.days
chain_all["adjusted"] = chain_all["name"].str.endswith("A")
chain = chain_all[~chain_all["adjusted"]].copy()
shibor = query.sql("SELECT * FROM curated.shibor WHERE date <= ? ORDER BY date", [END]).set_index("date")
spot = query.sql(
    """
    SELECT ts_code, trade_date, close FROM curated.fund_daily
    UNION ALL SELECT ts_code, trade_date, close FROM curated.index_daily
    """
).pivot_table(index="trade_date", columns="ts_code", values="close").sort_index()
UNDERLYING = {code: code.removeprefix("OP") for code in sorted(chain["opt_code"].unique())}
chain.groupby(["exchange", "opt_code"]).agg(
    rows=("ts_code", "size"), days=("trade_date", "nunique"), expiries=("maturity_date", "nunique"),
    adjusted_excluded=("adjusted", "sum"),
)

## 2. The calculation

In [ ]:
TENOR_DAYS = {"on": 1, "1w": 7, "2w": 14, "1m": 30, "3m": 90, "6m": 180, "9m": 270, "1y": 360}


def compute_rate(trade_date: str, days: int) -> float:
    """Continuously compounded rate for `days` from that day's SHIBOR (percent, ACT/360 simple)."""
    row = shibor.loc[:trade_date].iloc[-1]
    tenors = sorted(TENOR_DAYS.items(), key=lambda kv: kv[1])
    xs = [d for _, d in tenors]
    ys = [row[k] / 100 for k, _ in tenors]
    simple = float(np.interp(days, xs, ys))
    return math.log(1 + simple * days / 360) / (days / 365)


def select_terms(day_chain: pd.DataFrame) -> tuple[str, str]:
    """Near and next expiry: the two nearest with at least MIN_DAYS to expiry."""
    expiries = sorted(day_chain.loc[day_chain["days"] >= MIN_DAYS, "maturity_date"].unique())
    return expiries[0], expiries[1]


def build_quotes(term: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
    """Strike x (call, put) price table; settle replaces a missing/zero close."""
    price = term["close"].where(term["close"] > 0, term["settle"])
    used_settle = bool((price != term["close"]).any())
    quotes = term.assign(price=price).pivot_table(index="strike", columns="call_put", values=["price", "vol"])
    return quotes.sort_index(), used_settle


def compute_forward(quotes: pd.DataFrame, rate: float, t: float) -> tuple[float, float]:
    """F from put-call parity at the strike with the smallest |C - P|; K0 = largest strike <= F."""
    diff = (quotes[("price", "C")] - quotes[("price", "P")]).dropna()
    k_star = diff.abs().idxmin()
    forward = k_star + math.exp(rate * t) * diff[k_star]
    k0 = max(k for k in quotes.index if k <= forward)
    return forward, k0


def compute_variance(quotes: pd.DataFrame, forward: float, k0: float, rate: float, t: float, zero_rule: str) -> tuple[float, int]:
    """CBOE sigma^2 for one term and the number of strikes used."""
    def is_zero(k: float, side: str) -> bool:
        price, vol = quotes.at[k, ("price", side)], quotes.at[k, ("vol", side)]
        dead = pd.isna(price) or price <= 0
        return dead or (zero_rule == "price_or_volume" and (pd.isna(vol) or vol <= 0))

    selected: dict[float, float] = {k0: (quotes.at[k0, ("price", "C")] + quotes.at[k0, ("price", "P")]) / 2}
    for side, strikes in (("P", sorted((k for k in quotes.index if k < k0), reverse=True)),
                          ("C", sorted(k for k in quotes.index if k > k0))):
        zeros = 0
        for k in strikes:
            if is_zero(k, side):
                zeros += 1
                if zeros == 2:
                    break
                continue
            zeros = 0
            selected[k] = quotes.at[k, ("price", side)]
    ks = sorted(selected)
    total = 0.0
    for i, k in enumerate(ks):
        lo, hi = ks[max(i - 1, 0)], ks[min(i + 1, len(ks) - 1)]
        dk = (hi - lo) / (2 if 0 < i < len(ks) - 1 else 1)
        total += dk / k**2 * math.exp(rate * t) * selected[k]
    return 2 / t * total - (forward / k0 - 1) ** 2 / t, len(ks)


def compute_vix_day(day_chain: pd.DataFrame, trade_date: str, zero_rule: str = "price_or_volume") -> dict:
    """One row of the VIX panel for one trading day."""
    near, nxt = select_terms(day_chain)
    row, flags = {"trade_date": trade_date}, []
    for label, expiry in (("near", near), ("next", nxt)):
        term = day_chain[day_chain["maturity_date"] == expiry]
        days = int(term["days"].iloc[0])
        t = days * 1440 / N365
        rate = compute_rate(trade_date, days)
        quotes, used_settle = build_quotes(term)
        forward, k0 = compute_forward(quotes, rate, t)
        var, n = compute_variance(quotes, forward, k0, rate, t, zero_rule)
        flags += ["used_settle"] if used_settle else []
        row.update({f"days_{label}": days, f"rate_{label}": rate, f"f_{label}": forward,
                    f"k0_{label}": k0, f"var_{label}": var, f"n_options_{label}": n, f"t_{label}": t})
    n1, n2 = row["days_near"] * 1440, row["days_next"] * 1440
    w1, w2 = (n2 - N30) / (n2 - n1), (N30 - n1) / (n2 - n1)
    row["vix"] = 100 * math.sqrt((row["t_near"] * row["var_near"] * w1 + row["t_next"] * row["var_next"] * w2) * N365 / N30)
    row["quality_flag"] = ",".join(sorted(set(flags))) or "ok"
    return row

## 3. Check 1 - known answer on a synthetic chain

Price a chain with Black-Scholes at a flat 20% volatility; a correct VIX implementation must return ≈ 20.
The wide grid tests the formula; the narrow grid (a typical ETF chain's width, 2.65-3.60) shows how much strip truncation alone lowers the index.

In [ ]:
def price_black_scholes(s: float, k: float, t: float, r: float, sigma: float, side: str) -> float:
    d1 = (math.log(s / k) + (r + sigma**2 / 2) * t) / (sigma * math.sqrt(t))
    d2 = d1 - sigma * math.sqrt(t)
    n = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
    if side == "C":
        return s * n(d1) - k * math.exp(-r * t) * n(d2)
    return k * math.exp(-r * t) * n(-d2) - s * n(-d1)


def build_synthetic_chain(strikes: np.ndarray, sigma: float = 0.20, s: float = 3.04) -> pd.DataFrame:
    rows = []
    for expiry, days in (("20261023", 29), ("20261127", 64)):
        t = days * 1440 / N365
        r = compute_rate(END, days)
        for k in strikes:
            for side in ("C", "P"):
                rows.append({"maturity_date": expiry, "days": days, "strike": round(float(k), 4), "call_put": side,
                             "close": price_black_scholes(s, k, t, r, sigma, side), "settle": np.nan, "vol": 1.0})
    return pd.DataFrame(rows)


synthetic = pd.DataFrame([
    {"grid": "wide 1.50-5.00 step 0.05", **compute_vix_day(build_synthetic_chain(np.arange(1.50, 5.0001, 0.05)), END, "price")},
    {"grid": "typical width 2.65-3.60", **compute_vix_day(build_synthetic_chain(np.arange(2.65, 3.6001, 0.05)), END, "price")},
])
synthetic[["grid", "vix", "f_near", "k0_near", "n_options_near", "n_options_next"]]

## 4. VIX for every underlying and trading day

In [ ]:
def compute_vix_panel(zero_rule: str) -> pd.DataFrame:
    rows = []
    for (opt_code, trade_date), group in chain.groupby(["opt_code", "trade_date"]):
        try:
            row = compute_vix_day(group, trade_date, zero_rule)
        except (IndexError, ValueError, KeyError, ZeroDivisionError) as exc:
            row = {"trade_date": trade_date, "vix": np.nan, "quality_flag": f"failed: {type(exc).__name__}"}
        rows.append({"opt_code": opt_code, "underlying": UNDERLYING[opt_code], **row})
    return pd.DataFrame(rows)


vix = compute_vix_panel("price_or_volume")
vix["vix_price_rule"] = compute_vix_panel("price")["vix"].values
vix.groupby("opt_code").agg(days=("trade_date", "size"), failed=("vix", lambda s: int(s.isna().sum())),
                            vix_mean=("vix", "mean"), vix_min=("vix", "min"), vix_max=("vix", "max")).round(2)

## 5. Most recent VIX (2026-09-24) - the 300 family first

In [ ]:
latest = vix[vix["trade_date"] == vix["trade_date"].max()].copy()
latest["family_300"] = latest["opt_code"].isin(FAMILY_300)
latest.sort_values(["family_300", "vix"], ascending=[False, False])[
    ["trade_date", "opt_code", "underlying", "vix", "vix_price_rule", "days_near", "days_next",
     "f_near", "k0_near", "n_options_near", "n_options_next", "quality_flag"]
].round(3)

## 6. Check 2 - forward vs the underlying close

Put-call parity gives the market's forward; it must sit next to spot (`F/S` ≈ e^{(r−q)T}). Index options can trade at a discount to spot (dividends, hedging demand), so their band is wider.

In [ ]:
vix["spot"] = [spot.at[d, u] if (d in spot.index and u in spot.columns) else np.nan
               for d, u in zip(vix["trade_date"], vix["underlying"], strict=True)]
vix["f_over_spot"] = vix["f_near"] / vix["spot"]
vix.groupby("opt_code")["f_over_spot"].agg(["min", "max"]).round(4)

## 7. Check 3 - VIX vs at-the-money implied volatility

Invert Black-76 at `K0` (average of call and put IV) for both terms and interpolate to 30 days in variance. VIX integrates the whole smile, so it should sit near, usually slightly above, ATM IV.

In [ ]:
def compute_implied_vol(price: float, f: float, k: float, t: float, r: float, side: str) -> float:
    """Black-76 implied volatility by bisection."""
    n = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))

    def price_black(sigma: float) -> float:
        d1 = (math.log(f / k) + sigma**2 * t / 2) / (sigma * math.sqrt(t))
        d2 = d1 - sigma * math.sqrt(t)
        call = math.exp(-r * t) * (f * n(d1) - k * n(d2))
        return call if side == "C" else call - math.exp(-r * t) * (f - k)

    lo, hi = 1e-4, 3.0
    for _ in range(100):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if price_black(mid) < price else (lo, mid)
    return (lo + hi) / 2


def compute_atm_iv_30d(day_chain: pd.DataFrame, row: pd.Series) -> float:
    """ATM (K0) implied vol of both terms, interpolated to 30 days in variance."""
    expiries = sorted(day_chain.loc[day_chain["days"] >= MIN_DAYS, "maturity_date"].unique())
    var_t = []
    for label, expiry in (("near", expiries[0]), ("next", expiries[1])):
        quotes, _ = build_quotes(day_chain[day_chain["maturity_date"] == expiry])
        k0, f, t, r = row[f"k0_{label}"], row[f"f_{label}"], row[f"t_{label}"], row[f"rate_{label}"]
        ivs = [compute_implied_vol(quotes.at[k0, ("price", s)], f, k0, t, r, s) for s in ("C", "P")]
        var_t.append(np.mean(ivs) ** 2 * t)
    n1, n2 = row["days_near"] * 1440, row["days_next"] * 1440
    total = var_t[0] * (n2 - N30) / (n2 - n1) + var_t[1] * (N30 - n1) / (n2 - n1)
    return 100 * math.sqrt(total * N365 / N30)


groups = {key: g for key, g in chain.groupby(["opt_code", "trade_date"])}
vix["atm_iv_30d"] = [
    compute_atm_iv_30d(groups[(row["opt_code"], row["trade_date"])], row) if pd.notna(row["vix"]) else np.nan
    for _, row in vix.iterrows()
]
vix["vix_minus_atm"] = vix["vix"] - vix["atm_iv_30d"]
vix.groupby("opt_code")["vix_minus_atm"].agg(["min", "mean", "max"]).round(2)

## 8. Check 4 - VIX vs realized volatility of the underlying

20-day close-to-close realized vol (annualised, √252). Implied usually runs above realized (variance risk premium).

In [ ]:
realized = (np.log(spot).diff().rolling(20).std() * math.sqrt(252) * 100)
vix["realized_20d"] = [realized.at[d, u] if (d in realized.index and u in realized.columns) else np.nan
                       for d, u in zip(vix["trade_date"], vix["underlying"], strict=True)]
vix.groupby("opt_code")[["vix", "realized_20d"]].mean().round(2)

## 9. The 300 family over time

In [ ]:
theme = ChartTheme.from_mode("light")
colors = {"OP510300.SH": "#2a78d6", "OP159919.SZ": "#eb6834", "OP000300.SH": "#1baf7a"}
labels = {"OP510300.SH": "SSE 300ETF", "OP159919.SZ": "SZSE 300ETF", "OP000300.SH": "CSI 300 index (CFFEX)"}
fig = go.Figure()
last = vix[vix["trade_date"] == vix["trade_date"].max()].set_index("opt_code")["vix"]
label_shift = {code: 14 * (1 - rank) for rank, code in enumerate(last[FAMILY_300].sort_values(ascending=False).index)}
for opt_code in FAMILY_300:
    s = vix[vix["opt_code"] == opt_code].sort_values("trade_date")
    x = pd.to_datetime(s["trade_date"]).dt.strftime("%Y-%m-%d")
    fig.add_trace(go.Scatter(x=x, y=s["vix"], name=labels[opt_code], mode="lines+markers",
                             line={"color": colors[opt_code], "width": 2}, marker={"size": 8},
                             hovertemplate=labels[opt_code] + ": %{y:.2f}<extra></extra>"))
    fig.add_annotation(x=x.iloc[-1], y=s["vix"].iloc[-1], text=f"{labels[opt_code]} {s['vix'].iloc[-1]:.1f}",
                       showarrow=False, xanchor="left", xshift=8, yshift=label_shift[opt_code], font={"color": theme.text_secondary, "size": 11})
fig.update_layout(title="CSI 300 option VIX (CBOE method), 2026-08-03 to 2026-09-24", hovermode="x unified",
                  paper_bgcolor=theme.surface, plot_bgcolor=theme.surface, font={"color": theme.text_primary},
                  width=1000, height=480, margin={"l": 60, "r": 200, "t": 56, "b": 40},
                  legend={"orientation": "h", "y": -0.15})
fig.update_xaxes(type="category", nticks=10, gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
fig.update_yaxes(title_text="vol points", gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
fig.show()

## 10. Check summary

In [ ]:
ok = vix.dropna(subset=["vix"])
etf = ok[~ok["opt_code"].isin(["OP000016.SH", "OP000300.SH", "OP000852.SH"])]
ratio = ok.groupby("opt_code")["vix"].mean() / ok.groupby("opt_code")["realized_20d"].mean()
checks = pd.DataFrame([
    {"check": "1 synthetic flat 20% (wide grid)", "value": f"{synthetic['vix'].iloc[0]:.3f}",
     "pass rule": "|VIX - 20| < 0.25", "pass": abs(synthetic["vix"].iloc[0] - 20) < 0.25},
    {"check": "0 coverage", "value": f"{len(ok)}/{len(vix)} underlying-days computed",
     "pass rule": "no failed rows", "pass": len(ok) == len(vix)},
    {"check": "2 forward / spot, ETF options", "value": f"{etf['f_over_spot'].min():.4f} .. {etf['f_over_spot'].max():.4f}",
     "pass rule": "within 0.99 .. 1.01", "pass": bool(etf["f_over_spot"].between(0.99, 1.01).all())},
    {"check": "2b forward / spot, index options", "value": f"{ok.loc[~ok.index.isin(etf.index), 'f_over_spot'].min():.4f} .. {ok.loc[~ok.index.isin(etf.index), 'f_over_spot'].max():.4f}",
     "pass rule": "within 0.97 .. 1.01", "pass": bool(ok.loc[~ok.index.isin(etf.index), "f_over_spot"].between(0.97, 1.01).all())},
    {"check": "3 VIX - ATM IV (vol pts)", "value": f"{ok['vix_minus_atm'].min():.2f} .. {ok['vix_minus_atm'].max():.2f}",
     "pass rule": "every row within -2 .. +5", "pass": bool(ok["vix_minus_atm"].between(-2, 5).all())},
    {"check": "4 VIX / realized 20d (means)", "value": f"{ratio.min():.2f} .. {ratio.max():.2f}",
     "pass rule": "every underlying 0.5 .. 3", "pass": bool(ratio.between(0.5, 3).all())},
])
catalog.close()
checks

## 11. Where the checks miss their pre-set bands

The bands in §10 were fixed before looking at the results and are not widened after the fact. The rows outside them are listed here so the reason can be judged.

In [ ]:
outside_forward = etf[~etf["f_over_spot"].between(0.99, 1.01)]
outside_atm = ok[~ok["vix_minus_atm"].between(-2, 5)]
print("forward/spot outside 0.99..1.01 (ETF options):")
print(outside_forward.groupby("opt_code").agg(rows=("trade_date", "size"), worst=("f_over_spot", "min"),
      mean_days_near=("days_near", "mean")).round(4).to_string())
print()
print("VIX - ATM IV outside -2..+5:")
print(outside_atm[["trade_date", "opt_code", "vix", "atm_iv_30d", "vix_minus_atm", "n_options_near", "n_options_next"]].round(2).to_string(index=False))